# Agentic RAG / ReAct — 도구 + 추론 루프 — 복원본 (대화 기반 재구성)

> ⚠️ 원본이 저장 누락으로 0바이트가 돼서, Claude와의 학습 대화에 남은 코드·구조로 **재구성**한 거야.
> 네가 그때 짠 원본 그대로는 아니니 **한 번 검토하고 직접 재실행**해. `TODO/확인` 자리는 네 데이터/환경에 맞춰 채우면 돼.
> LLM 생성이 필요한 셀은 무료 Groq API 키가 필요해 (OpenAI 크레딧 소진 상태).

## 개념
일반 RAG는 "무조건 검색 → 생성"의 **고정 단계**.
Agentic RAG는 검색을 **에이전트가 부를지 말지 스스로 결정하는 도구(tool)**로 바꿈.
ReAct = Reasoning + Acting: (생각 → 도구 호출 → 관찰 → 다시 생각) 루프.
핵심은 "부를지 결정" 그 자체 — "안녕" 같은 입력엔 검색을 *안* 부르는 게 보이면 성공.

In [ ]:
# Groq 무료 API 필요
from langchain_groq import ChatGroq
from langchain.tools.retriever import create_retriever_tool
from langgraph.prebuilt import create_react_agent

# (앞 Chroma 노트북의 vectorstore 재사용 가정)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

retriever_tool = create_retriever_tool(
    retriever,
    name="knowledge_search",
    description="LLM/RAG/파인튜닝 등 개념 질문일 때 관련 문서를 검색한다.",
)

llm = ChatGroq(model="llama-3.3-70b-versatile")
agent = create_react_agent(llm, tools=[retriever_tool])

In [ ]:
# 검색을 부르는 질문 vs 안 부르는 인사 비교
resp = agent.invoke({"messages": [("user", "RAG가 뭐야?")]})
for m in resp["messages"]:
    print(m)          # 도구 호출 결정 흐름이 보임 (tool call → 결과 → 답변)

# TODO/확인: "안녕"으로 invoke하면 knowledge_search를 호출하지 '않는' 것 확인
# resp2 = agent.invoke({"messages": [("user", "안녕")]})